In [1]:
import os
import pandas as pd
import numpy as np


# Paths
PROCESSED_FILE = os.path.join("..", "data", "processed", "cardio_onc_prostate_03cleaned.csv")
BASE_FILE = os.path.join("..", "data", "processed", "cardio_onc_prostate_06_broad_clean.csv")
df = pd.read_csv(PROCESSED_FILE)

print(df.shape)
df.head()

(239, 50)


,unique_patient_id,ethnicity,nht_auth_date,nht_start_date,bmi,specific_nht_used,age,adt_start_date,adt_agent,hx_smoking,...,cards_prior,cards_post,cards_referral,diet_counseling,exercise_counseling,echo_ordered,ecg_done,non_onc_provider,has_pcp_duplicate,prescribing_provider
0,1.0,NaN,2022-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MEI, ERIKA A"
1,2.0,NaN,2022-01-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MEI, ERIKA A"
2,3.0,1.0,2022-01-12,2022-02-01,25.99,Darolutamide,73.0,2018-01-26,Lupron,1.0,...,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,2.0,"MEI, ERIKA A"
3,4.0,3.0,2022-01-14,2022-02-14,22.55,Apalutamide,93.0,2021-12-01,Bicalutamide,0.0,...,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,3.0,"UCHIO, EDWARD M"
4,5.0,NaN,2022-01-19,NaN,NaN,Abiraterone,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"DANESHVAR, MICHAEL A"


Preserve Raw Medication Intensity + Create Binary Versions

In [2]:
post_cols = ['bp_meds_post', 'lipid_meds_post', 'dm_meds_post']

# Create binary versions WITHOUT overwriting originals
for col in post_cols:
    df[col + "_binary"] = df[col].where(df[col].isna(), (df[col] != 0).astype(int))

# Check
for col in post_cols + [c + "_binary" for c in post_cols]:
    print(f"\n{col}")
    print(df[col].value_counts(dropna = False))


bp_meds_post
bp_meds_post
0.0    165
NaN     37
1.0     25
2.0     11
3.0      1
Name: count, dtype: int64

lipid_meds_post
lipid_meds_post
0.0    175
NaN     36
1.0     19
2.0      9
Name: count, dtype: int64

dm_meds_post
dm_meds_post
0.0    161
NaN     37
2.0     25
1.0     16
Name: count, dtype: int64

bp_meds_post_binary
bp_meds_post_binary
0.0    165
NaN     37
1.0     37
Name: count, dtype: int64

lipid_meds_post_binary
lipid_meds_post_binary
0.0    175
NaN     36
1.0     28
Name: count, dtype: int64

dm_meds_post_binary
dm_meds_post_binary
0.0    161
1.0     41
NaN     37
Name: count, dtype: int64


Create Target Variable (at_risk)

In [3]:
df['at_risk'] = df[
    ['bp_meds_post_binary', 'lipid_meds_post_binary', 'dm_meds_post_binary']
].max(axis = 1)

print(df['at_risk'].value_counts(dropna = False))

at_risk
0.0    124
1.0     79
NaN     36
Name: count, dtype: int64


Standardize NHT Names

In [4]:
col = 'specific_nht_used'
df[col] = df[col].str.strip().str.lower()

mapping = {
    'abiraterone': 'Abiraterone',
    'abiaterone': 'Abiraterone',
    'zytiga': 'Abiraterone',
    'darolutamide': 'Darolutamide',
    'enzalutamide': 'Enzalutamide',
    'enzaltamide': 'Enzalutamide',
    'apalutamide': 'Apalutamide',
    'apalutamide to darolutamide': 'Apalutamide'
}

df[col] = df[col].replace(mapping)

print(df[col].value_counts(dropna = False))

specific_nht_used
Abiraterone     104
Darolutamide     91
Enzalutamide     24
Apalutamide      14
NaN               6
Name: count, dtype: int64


Standardize ADT Agents

In [5]:
col = 'adt_agent'
df[col] = df[col].str.strip().str.lower()

mapping = {
    'orgovyx': 'Orgovyx',
    'orogovyx': 'Orgovyx',

    'lupron': 'Lupron',
    'lupron depot': 'Lupron',
    'leupron': 'Lupron',

    'bicalutamide': 'Bicalutamide',
    'casodex': 'Bicalutamide',

    'firmagon': 'Firmagon',

    'firmagon to lupron': 'Lupron',
    'firmagon to orgovyx': 'Orgovyx',
    'lupron to orgovyx': 'Orgovyx',
    'bicalutamide to lupron': 'Lupron'
}

df[col] = df[col].replace(mapping)

print(df[col].value_counts(dropna = False))

adt_agent
Orgovyx                              80
Lupron                               55
NaN                                  44
Bicalutamide                         27
Firmagon                             18
bicalutamide to lupron to orgovyx     2
lupron, orgovyx (d/c)                 2
lupron + casodex to lupron            2
bicalutamide, then lupron             1
orgovyx and nubeqa                    1
lupron, firmagon                      1
lupron + bicalutamide                 1
bicalutamide + lupron                 1
casodex to lpron                      1
casodex to lupron                     1
casodex to firmagon                   1
lupon                                 1
Name: count, dtype: int64


Standardize Binary Encodings (WITHOUT collapsing information)

In [6]:
binary_1_yes = [
    'bb_prior', 'ace_arb_prior', 'hx_hld',
    'hx_high_tg', 'statin_prior', 'other_lipid_prior',
    'lipid_panel_checked', 'hx_cad', 'hx_mi_stent',
    'hx_chf', 'hx_arrhythmia', 'hx_carotid',
    'hx_pad', 'hx_cva', 'hx_dm2', 'glucose_over_200',
    'asa_use', 'cards_prior', 'cards_post', 'cards_referral',
    'diet_counseling', 'exercise_counseling',
    'echo_ordered', 'ecg_done'
]

for col in binary_1_yes:
    df[col] = df[col].where(df[col].isna(), (df[col] == 1).astype(int))

Fix Special Columns

In [7]:
# has_pcp flip
df['has_pcp'] = df['has_pcp'].where(df['has_pcp'].isna(), (df['has_pcp'] != 1).astype(int))

# family history
df['family_hx_hd'] = df['family_hx_hd'].replace({
    1: 1,
    2: 0,
    3: np.nan,
    0: np.nan
})

# NA-zero columns
df[['hx_htn', 'on_insulin', 'a1c_checked']] = df[
    ['hx_htn', 'on_insulin', 'a1c_checked']
].replace({1: 1, 2: 0, 0: np.nan})

Ethnicity Mapping

In [8]:
ethnicity_map = {
    1: "Caucasian",
    2: "Hispanic",
    3: "Asian",
    4: "Black",
    5: "Other"
}

df['ethnicity'] = df['ethnicity'].map(ethnicity_map)

Dates (ONLY parsing, no modeling features yet)

In [9]:
date_cols = ["nht_auth_date", "nht_start_date", "adt_start_date"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors = "coerce")

Numeric Casting

In [10]:
print(df.columns.tolist())

['unique_patient_id', 'ethnicity', 'nht_auth_date', 'nht_start_date', 'bmi', 'specific_nht_used', 'age', 'adt_start_date', 'adt_agent', 'hx_smoking', 'family_hx_hd', 'hx_htn', 'sbp', 'dbp', 'bp_meds_prior', 'bb_prior', 'ace_arb_prior', 'bp_meds_post', 'has_pcp', 'hx_hld', 'hx_high_tg', 'statin_prior', 'other_lipid_prior', 'lipid_meds_post', 'lipid_panel_checked', 'ascvd_10yr', 'hx_cad', 'hx_mi_stent', 'hx_chf', 'hx_arrhythmia', 'hx_carotid', 'hx_pad', 'hx_cva', 'hx_dm2', 'dm_noninsulin', 'on_insulin', 'dm_meds_post', 'a1c_checked', 'glucose_over_200', 'asa_use', 'cards_prior', 'cards_post', 'cards_referral', 'diet_counseling', 'exercise_counseling', 'echo_ordered', 'ecg_done', 'non_onc_provider', 'has_pcp_duplicate', 'prescribing_provider', 'bp_meds_post_binary', 'lipid_meds_post_binary', 'dm_meds_post_binary', 'at_risk']


In [11]:
num_cols = ["bmi", "age", "sbp", "dbp", "ascvd_10yr"]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors = "coerce")

Keep BOTH Raw + Engineered Features

In [12]:
# Safe boolean OR
df["lifestyle_counseling"] = (
    (df["diet_counseling"] == 1) | (df["exercise_counseling"] == 1)
).astype("Int8")

In [13]:
def diabetes_severity(row):
    if pd.isna(row['hx_dm2']):
        return pd.NA
    if row['hx_dm2'] == 0:
        return 0
    if row['on_insulin'] == 1:
        return 2
    return 1


df['dm_severity'] = df.apply(diabetes_severity, axis = 1).astype("Int8")

print(df['dm_severity'].value_counts(dropna = False))

dm_severity
0       141
1        46
<NA>     38
2        14
Name: count, dtype: Int64


Label Encoding (for modeling flexibility)

In [14]:
from sklearn.preprocessing import LabelEncoder


cat_cols = ["ethnicity", "specific_nht_used", "adt_agent", "prescribing_provider"]

le_map = {}

for col in cat_cols:
    le = LabelEncoder()
    mask = df[col].notna()
    df.loc[mask, col + "_enc"] = le.fit_transform(df.loc[mask, col].astype(str))
    df[col + "_enc"] = df[col + "_enc"].astype("Int16")
    le_map[col] = le

In [15]:
print("Missing values (kept intentionally for flexibility):")
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values (kept intentionally for flexibility):
ethnicity                    32
nht_auth_date                 5
nht_start_date               29
bmi                          32
specific_nht_used             6
age                          32
adt_start_date               46
adt_agent                    44
hx_smoking                   34
family_hx_hd                103
hx_htn                       39
sbp                          88
dbp                          88
bp_meds_prior                36
bb_prior                     36
ace_arb_prior                36
bp_meds_post                 37
has_pcp                      37
hx_hld                       36
hx_high_tg                   37
statin_prior                 36
other_lipid_prior            36
lipid_meds_post              36
lipid_panel_checked          36
ascvd_10yr                    2
hx_cad                       37
hx_mi_stent                  36
hx_chf                       36
hx_arrhythmia                36
hx_carotid         

In [16]:
print(f"Shape: {df.shape}")
print(df.dtypes)
df.head()

Shape: (239, 60)
unique_patient_id                  float64
ethnicity                           object
nht_auth_date               datetime64[ns]
nht_start_date              datetime64[ns]
bmi                                float64
specific_nht_used                   object
age                                float64
adt_start_date              datetime64[ns]
adt_agent                           object
hx_smoking                         float64
family_hx_hd                       float64
hx_htn                             float64
sbp                                float64
dbp                                float64
bp_meds_prior                      float64
bb_prior                           float64
ace_arb_prior                      float64
bp_meds_post                       float64
has_pcp                            float64
hx_hld                             float64
hx_high_tg                         float64
statin_prior                       float64
other_lipid_prior                  fl

,unique_patient_id,ethnicity,nht_auth_date,nht_start_date,bmi,specific_nht_used,age,adt_start_date,adt_agent,hx_smoking,...,bp_meds_post_binary,lipid_meds_post_binary,dm_meds_post_binary,at_risk,lifestyle_counseling,dm_severity,ethnicity_enc,specific_nht_used_enc,adt_agent_enc,prescribing_provider_enc
0,1.0,NaN,2022-01-09,NaT,NaN,NaN,NaN,NaT,NaN,NaN,...,NaN,NaN,NaN,NaN,0,<NA>,<NA>,<NA>,<NA>,27
1,2.0,NaN,2022-01-11,NaT,NaN,NaN,NaN,NaT,NaN,NaN,...,NaN,NaN,NaN,NaN,0,<NA>,<NA>,<NA>,<NA>,27
2,3.0,Caucasian,2022-01-12,2022-02-01,25.99,Darolutamide,73.0,2018-01-26,Lupron,1.0,...,0.0,0.0,0.0,0.0,0,0,2,2,2,27
3,4.0,Asian,2022-01-14,2022-02-14,22.55,Apalutamide,93.0,2021-12-01,Bicalutamide,0.0,...,0.0,0.0,0.0,0.0,0,0,0,1,0,42
4,5.0,NaN,2022-01-19,NaT,NaN,Abiraterone,NaN,NaT,NaN,NaN,...,NaN,NaN,NaN,NaN,0,<NA>,<NA>,0,<NA>,7


In [17]:
df.to_csv(BASE_FILE, index = False)
print(f"Saved base dataset to: {BASE_FILE}")

Saved base dataset to: ../data/processed/cardio_onc_prostate_06_broad_clean.csv
